# 04 — Rekap dan Perbandingan (bahan presentasi)

Urutan: **data → desain → hasil utama → efek skenario → per kelas → kontribusi transfer learning → XAI → efisiensi → kesimpulan**.

Notebook ini hanya membaca hasil di `outputs/`, tanpa training dan tanpa inferensi baru, sehingga aman dijalankan kapan saja. Model yang belum selesai dilewati. Semua gambar disimpan ke `outputs/paper/` dalam format PNG 300 dpi (untuk slide) dan PDF (untuk paper). Poin bahasan di bawah setiap gambar diisi otomatis dari angka hasil.

Warna skenario: S1 abu-abu, S2 biru, S3 oranye.

**Versi R15.** Angka resmi berasal dari Colab L4 (satu akun per backbone), digabung di `outputs_colab_r15`. Model utama paper adalah **model lengkap**: CNN + Beat-Level Specialist + fusi regresi logistik + kunci VT. Hasil CNN saja dilaporkan sebagai komponen (ablasi). Bagian B1 dan setelahnya adalah hasil CNN saja.


In [ ]:
import os
from pathlib import Path as _P
_folder = next(p for p in [_P.cwd(), *_P.cwd().parents] if (p / "v4_pipeline.py").exists() or (p / "eksperimen_v4_svdb" / "v4_pipeline.py").exists())
_folder = _folder if (_folder / "v4_pipeline.py").exists() else _folder / "eksperimen_v4_svdb"
os.environ["V4_OUTPUT_ROOT"] = str(_folder / "outputs_colab_r15")
print("Hasil dibaca dari:", os.environ["V4_OUTPUT_ROOT"])
import sys, json, time
from pathlib import Path
FOLDER = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v4_pipeline.py").exists() or (p / "eksperimen_v4_svdb" / "v4_pipeline.py").exists())
FOLDER = FOLDER if (FOLDER / "v4_pipeline.py").exists() else FOLDER / "eksperimen_v4_svdb"
sys.path.insert(0, str(FOLDER))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
import v4_runner as R
import v4_figures as F
print("Folder eksperimen:", FOLDER)

R.status()


## A. Data dan desain


### A1. Alur penelitian


In [ ]:
F.fig_flow(); plt.show()


### A2. Komposisi dataset


In [ ]:
tables = F.dataset_tables(); display(tables["development"]); display(tables["holdout"]); display(tables["folds"])
F.fig_dataset(); plt.show()


### A3. Contoh citra per kelas (citra global 10 detik + crop lokal per beat)


In [ ]:
F.fig_examples(); plt.show()


## B. Hasil utama


### B0. Model lengkap (model utama paper)
Holdout dievaluasi sekali dengan model final pada E*. Specialist dan fusi dikunci dari data development; holdout tidak dipakai untuk seleksi.


In [ ]:
display(F.lengkap_table("holdout")); display(F.lengkap_table("cv"))


### B0.1 CNN saja vs model lengkap (holdout)
Menunjukkan sumbangan Beat-Level Specialist + fusi + kunci VT di atas CNN.


In [ ]:
display(F.cnn_vs_lengkap()); F.fig_lengkap_bars(); plt.show()


### B0.2 Uji efek skenario pada model lengkap dan CNN saja
Bootstrap pasien berpasangan (2.000 ulangan), koreksi Holm atas 8 metrik.


In [ ]:
ef = F.lengkap_effects("lengkap"); display(ef[ef.Metrik.isin(["macro_F1", "F1_S", "recall_S", "recall_VT"])])
ec = F.lengkap_effects("cnn"); display(ec[ec.Metrik.isin(["macro_F1", "F1_S", "recall_S", "recall_VT"])])


### B1. Tabel utama: CV (rerata ± SD pada E*) dan holdout (CI 95% bootstrap pasien)


In [ ]:
F.main_table()


### B2. Macro-F1 per backbone dan skenario (garis merah = target 0,80)


In [ ]:
F.fig_macro_bars(); plt.show()
display(Markdown(F.talking_points_main()))


### B3. Confusion matrix holdout
Setiap sel berisi **jumlah window yang ditebak (persentase terhadap jumlah kelas sebenarnya)**, jadi persen pada diagonal adalah recall. Warna mengikuti persentase. Support (`n`) tertulis di kanan, precision (`P`) di bawah.


#### EfficientNet-B0


In [ ]:
F.fig_confusion("efficientnet_b0"); plt.show()
display(Markdown(F.talking_points_confusion("efficientnet_b0")))


#### MobileNetV2


In [ ]:
F.fig_confusion("mobilenet_v2"); plt.show()
display(Markdown(F.talking_points_confusion("mobilenet_v2")))


#### ShuffleNetV2 x1.0


In [ ]:
F.fig_confusion("shufflenet_v2_x1_0"); plt.show()
display(Markdown(F.talking_points_confusion("shufflenet_v2_x1_0")))


### B4. Confusion matrix CV (gabungan fold validasi pada E*), sebagai pembanding holdout


In [ ]:
F.fig_confusion("efficientnet_b0", "cv"); plt.show()


In [ ]:
F.fig_confusion("mobilenet_v2", "cv"); plt.show()


In [ ]:
F.fig_confusion("shufflenet_v2_x1_0", "cv"); plt.show()


### B5. Perubahan confusion matrix: S2−S1 dan S3−S2
Angka dalam poin persen baris. Biru berarti naik, merah berarti turun. Diagonal yang biru berarti recall kelas itu naik; sel di luar diagonal menunjukkan ke kelas mana salah tebaknya berpindah.


In [ ]:
F.fig_confusion_diff("efficientnet_b0"); plt.show()


In [ ]:
F.fig_confusion_diff("mobilenet_v2"); plt.show()


In [ ]:
F.fig_confusion_diff("shufflenet_v2_x1_0"); plt.show()


### B6. Kurva validasi dan epoch final bersama E*


In [ ]:
F.fig_training_curves("efficientnet_b0"); plt.show()


In [ ]:
F.fig_training_curves("mobilenet_v2"); plt.show()


In [ ]:
F.fig_training_curves("shufflenet_v2_x1_0"); plt.show()


## C. Efek skenario (inti judul)


### C1. Selisih berpasangan: S2−S1 (class weighting) dan S3−S2 (fine-tuning)
CV: selisih per fold. Holdout: selisih dengan bootstrap pasien berpasangan, memakai pasien yang sama untuk kedua model.


In [ ]:
F.scenario_effects()


In [ ]:
F.fig_scenario_effects(); plt.show()
display(Markdown(F.talking_points_effects()))


### C2. Heatmap F1 per kelas


In [ ]:
F.fig_f1_heatmap("holdout"); F.fig_f1_heatmap("cv"); plt.show()


## D. Analisis per kelas (fokus kelas S)


### D1. Precision, recall, sensitivity, dan specificity per kelas (holdout)


In [ ]:
display(F.per_class_table().round(3)); F.fig_precision_recall(); plt.show()


### D2. ROC per kelas, model terbaik setiap backbone


In [ ]:
for arch in F.ARCHS:
    best = F.best_scenario(arch)
    if best:
        F.fig_roc(arch, best)
plt.show()


### D3. Recall kelas S per pasien holdout
Menunjukkan apakah deteksi S merata antar pasien atau hanya ditopang oleh sebagian pasien.


In [ ]:
F.fig_s_patients(); plt.show()


### D4. F1 kelas S dan V per database (mitdb vs svdb)


In [ ]:
F.fig_per_database(); plt.show()


## E. Kontribusi transfer learning (jalur citra) vs fitur tabular
Knock-out pada holdout: embedding citra diganti rerata data latih, dan/atau fitur tabular diganti 0. Porsi kontribusi dihitung dengan Shapley dua pemain.


In [ ]:
display(F.modality_table()); F.fig_modality(); plt.show()


## F. XAI: model dasar (S2) vs fine-tuning (S3)
Pemeriksaan kualitatif fokus model, bukan validasi klinis. Merah berarti atribusi mendukung kelas sebenarnya. Penanda di bawah citra: hijau untuk gelombang P, ungu untuk QRS.


In [ ]:
F.fig_xai_gallery("efficientnet_b0"); plt.show()


In [ ]:
F.fig_xai_gallery("mobilenet_v2"); plt.show()


In [ ]:
F.fig_xai_gallery("shufflenet_v2_x1_0"); plt.show()


### F2. Konsentrasi atribusi per segmen
Rasio massa |SHAP| terhadap luas segmen. Nilai > 1 berarti atribusi terkonsentrasi di segmen itu. Ekspektasi klinis: S → P/RR, V → QRS.


In [ ]:
display(F.xai_segment_table()); F.fig_xai_segments("lokal"); F.fig_xai_segments("global"); plt.show()


### F3. SHAP fitur tabular berkelompok


In [ ]:
F.fig_xai_tabular(); plt.show()


## G. Efisiensi (dari notebook 05, RTX 4050 lokal)


In [ ]:
display(F.efficiency_table()); F.fig_tradeoff(); plt.show()


## H. Ringkasan: jawaban rumusan masalah


In [ ]:
F.answers_table()


In [ ]:
print("Semua gambar dan tabel:", F.paper_dir()); sorted(p.name for p in F.paper_dir().iterdir())
